# **Lab 8 - Agency & Autonomy Part 1**
<p>COMP4146/7125 Prompt Engineering for Generative AI<br/>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>

## Lab Overview

**Audience:** students who have already finished Topic 7 on controlling answer structure and reasoning cost.

This lab focuses on four ideas:
- tool use;
- ReAct;
- context assembly;
- the agent loop.

All live model-based cells use one backend throughout: **`minimax-m2.5:cloud`** through Ollama.

**By the end of this notebook, you should be able to:**
1. explain why chat models need tools, external context, and application control to complete real tasks;
2. define tools that are easier for both humans and models to understand;
3. use `minimax-m2.5:cloud` through Ollama at least once in each major part of the lab;
4. trace a **Thought -> Action -> Observation -> Finish** loop in a ReAct-style agent;
5. assemble agent context from a preamble, prior conversation, current exchange, and artifacts;
6. identify where the application, not the prompt, must enforce approvals and surface tool activity to the user.

### Outline
1. Setup and backend check
2. Tool usage and safe dispatch
3. ReAct
4. Context for task-based interactions
5. The conversational agent loop

## Setup

This notebook combines:
- deterministic Python mini-environments for stable teaching examples;
- live Ollama calls using **`minimax-m2.5:cloud`** as the single LLM backend.

Before the live cells, make sure Ollama is installed, reachable, and configured to access `minimax-m2.5:cloud`. Cloud model usage on Ollama requires `ollama signin`.

If the backend is unavailable, the deterministic cells will still run and the live cells will print a short diagnostic instead of failing.

In [1]:
from __future__ import annotations

import json
import re
from copy import deepcopy
from textwrap import dedent


def show_json(obj):
    # Pretty-print nested data structures.
    print(json.dumps(obj, indent=2, ensure_ascii=True))


def count_words(text):
    # Cheap token-like word count for quick notebook diagnostics.
    return len(re.findall(r"\w+|[^\w\s]", text))


def short(text, limit=120):
    # Shorten a long string so printed outputs stay compact.
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def to_builtin(obj):
    # Convert pydantic-style responses to plain Python values.
    if hasattr(obj, "model_dump"):
        return to_builtin(obj.model_dump())
    if isinstance(obj, dict):
        return {key: to_builtin(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(value) for value in obj]
    return obj

In [2]:
try:
    import ollama

    OLLAMA_IMPORT_ERROR = None
    OLLAMA_CLIENT = ollama.Client()
except Exception as exc:
    ollama = None
    OLLAMA_CLIENT = None
    OLLAMA_IMPORT_ERROR = repr(exc)

TARGET_OLLAMA_MODEL = "minimax-m2.5:cloud"


def get_ollama_status():
    # Check whether the local Ollama client can reach the configured backend.
    if OLLAMA_CLIENT is None:
        return False, f"Python import failed: {OLLAMA_IMPORT_ERROR}"
    try:
        OLLAMA_CLIENT.ps()
        return True, "Connected to Ollama."
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


OLLAMA_READY, OLLAMA_STATUS = get_ollama_status()


def extract_message_text(chat_response):
    # Extract assistant text from an Ollama chat response.
    data = to_builtin(chat_response)
    return (data.get("message", {}) or {}).get("content")


def extract_tool_calls(chat_response):
    # Extract tool calls from an Ollama chat response.
    data = to_builtin(chat_response)
    return data.get("message", {}).get("tool_calls") or []


# Call Ollama chat API with LLMs which are compatible with tools, more can be found at https://ollama.com/blog/tool-support
def run_live_chat(messages, tools=None, temperature=0):
    # Run one live chat request and return either data or a short error string.
    if not OLLAMA_READY:
        return None, OLLAMA_STATUS
    try:
        response = OLLAMA_CLIENT.chat(
            model=TARGET_OLLAMA_MODEL,
            messages=messages,
            tools=tools,
            options={"temperature": temperature},
        )
        return to_builtin(response), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

### Backend Check

This notebook uses one live backend: `minimax-m2.5:cloud` through Ollama. Keeping the backend fixed makes the live behavior easier to compare across the notebook.

In [3]:
print("Lab 8 helpers loaded.")
print("Approx word count for this sentence:", count_words("Lab 8 helpers loaded."))
print()
print("Target backend:", TARGET_OLLAMA_MODEL)
print("Ollama ready:", OLLAMA_READY)
print("Status:", OLLAMA_STATUS)
if not OLLAMA_READY:
    print("Live cells will print a short diagnostic instead of failing.")

Lab 8 helpers loaded.
Approx word count for this sentence: 5

Target backend: minimax-m2.5:cloud
Ollama ready: True
Status: Connected to Ollama.


---

## Part 1. Tool Usage and Safe Dispatch

Bare chat models need tools when they must read live state or act in the world.

In this part we will:
- compare bloated vs focused tool definitions;
- build a tiny thermostat tool registry;
- ask `minimax-m2.5:cloud` to plan one tool call;
- enforce approval rules in application code rather than in prompt text.

### 1.1 Tool definitions should be simple enough to route correctly

Use four design rules:
1. expose only the tools needed for the current task;
2. use names that read clearly to humans;
3. keep definitions short and concrete;
4. keep arguments small and manageable.

The next cell contrasts one bloated tool with a smaller task-oriented toolset.

In [4]:
# One oversized tool tries to do too many unrelated things at once.
bloated_tool = {
    "type": "function",
    "function": {
        "name": "manage_student_portal",
        "description": (
            "Unified endpoint for balance lookups, password resets, profile updates, "
            "course drops, and transcript processing."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string"},
                "student_id": {"type": "string"},
                "new_email": {"type": "string"},
                "new_password": {"type": "string"},
                "course_code": {"type": "string"},
                "reason": {"type": "string"},
                "refund_amount": {"type": "number"},
                "include_unofficial_transcript": {"type": "boolean"},
            },
        },
    },
}

In [5]:
# A focused toolset partitions the task into smaller, easier routing decisions.
focused_tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_fee_balance",
            "description": "Return the current tuition balance for one student.",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "reset_student_password",
            "description": "Reset one student's portal password.",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "drop_course",
            "description": "Drop one student from one course after approval.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string"},
                    "course_code": {"type": "string"},
                },
            },
        },
    },
]

In [6]:
def definition_summary(tool):
    # Summarize one tool so the contrast is easy to read.
    props = tool.get("function", {}).get("parameters", {}).get("properties", {})
    return {
        "name": tool["function"]["name"],
        "description_chars": len(tool["function"].get("description", "")),
        "top_level_args": len(props),
        "arg_names": list(props.keys()),
    }


print("Bloated tool summary:")
show_json(definition_summary(bloated_tool))
print()
print("Focused tool summaries:")
for tool in focused_tools:
    show_json(definition_summary(tool))

Bloated tool summary:
{
  "name": "manage_student_portal",
  "description_chars": 112,
  "top_level_args": 8,
  "arg_names": [
    "action",
    "student_id",
    "new_email",
    "new_password",
    "course_code",
    "reason",
    "refund_amount",
    "include_unofficial_transcript"
  ]
}

Focused tool summaries:
{
  "name": "lookup_fee_balance",
  "description_chars": 51,
  "top_level_args": 1,
  "arg_names": [
    "student_id"
  ]
}
{
  "name": "reset_student_password",
  "description_chars": 36,
  "top_level_args": 1,
  "arg_names": [
    "student_id"
  ]
}
{
  "name": "drop_course",
  "description_chars": 48,
  "top_level_args": 2,
  "arg_names": [
    "student_id",
    "course_code"
  ]
}


**Interpretation**
- The bloated tool mixes many unrelated actions.
- The focused tools expose clearer routing choices.
- What is easier for a human to understand is usually easier for an LLM to use correctly as well.

### 1.2 The application executes tools and appends the trace

We will use two thermostat tools:
- `get_room_temp()`: read-only;
- `set_room_temp(temp)`: changes state.

The model can request these tools, but the application still owns execution, argument checks, error handling, and approval decisions.

In [7]:
# This small mutable state stands in for an external system.
thermostat_state = {"temp": 74, "original_temp": 74}


def get_room_temp():
    # Return the current room temperature as a string.
    return str(thermostat_state["temp"])


def set_room_temp(temp):
    # Update the room temperature after basic application-side checks.
    if not isinstance(temp, int):
        raise TypeError("`temp` must be an integer in Fahrenheit.")
    if temp < 50 or temp > 86:
        raise ValueError("Requested temperature is outside the safe demo range (50-86F).")
    thermostat_state["temp"] = temp
    return "DONE"

In [8]:
# These are the tool definitions shown to the model.
thermostat_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_room_temp",
            "description": "Get the ambient room temperature in Fahrenheit.",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_room_temp",
            "description": "Set the ambient room temperature in Fahrenheit.",
            "parameters": {
                "type": "object",
                "properties": {
                    "temp": {
                        "type": "integer",
                        "description": "Desired room temperature in Fahrenheit.",
                    }
                },
                "required": ["temp"],
            },
        },
    },
]

available_functions = {
    "get_room_temp": get_room_temp,
    "set_room_temp": set_room_temp,
}
dangerous_tools = {"set_room_temp"}


In [9]:
def execute_tool_call(tool_call, approved=False):
    # Execute one requested tool call with approval and error handling.
    name = tool_call["function"]["name"]
    arguments = tool_call["function"].get("arguments", {})

    if name not in available_functions:
        return {
            "status": "error",
            "role": "tool",
            "name": name,
            "content": f"Unknown tool: {name}",
        }

    if name in dangerous_tools and not approved:
        return {
            "status": "awaiting_approval",
            "tool": name,
            "arguments": arguments,
            "content": "Approval required before executing a state-changing tool.",
        }

    try:
        result = available_functions[name](**arguments)
        return {"status": "ok", "role": "tool", "name": name, "content": result}
    except Exception as exc:
        return {
            "status": "error",
            "role": "tool",
            "name": name,
            "content": f"ERROR: {type(exc).__name__}: {exc}",
        }

In [10]:
# These three calls illustrate the difference between read-only, gated, and approved execution.
read_call = {"function": {"name": "get_room_temp", "arguments": {}}}
write_call = {"function": {"name": "set_room_temp", "arguments": {"temp": 76}}}

print("Read-only call:")
show_json(execute_tool_call(read_call))
print()
print("Dangerous call without approval:")
show_json(execute_tool_call(write_call))
print()
print("Dangerous call after approval:")
show_json(execute_tool_call(write_call, approved=True))
print()
print("Updated thermostat state:", thermostat_state)

Read-only call:
{
  "status": "ok",
  "role": "tool",
  "name": "get_room_temp",
  "content": "74"
}

Dangerous call without approval:
{
  "status": "awaiting_approval",
  "tool": "set_room_temp",
  "arguments": {
    "temp": 76
  },
  "content": "Approval required before executing a state-changing tool."
}

Dangerous call after approval:
{
  "status": "ok",
  "role": "tool",
  "name": "set_room_temp",
  "content": "DONE"
}

Updated thermostat state: {'temp': 76, 'original_temp': 74}


### 1.3 Live Ollama tool planning with `minimax-m2.5:cloud`

We now ask `minimax-m2.5:cloud` to plan a thermostat turn using the same tool definitions.

Read the result carefully:
- did it choose the right tool?
- did it produce sensible arguments?
- what still needs to be validated before execution?

In [11]:
part1_messages = [
    {
        "role": "system",
        "content": (
            "You are a careful home assistant. Use tools when needed. "
            "If the user asks to change temperature, check the current temperature first."
        ),
    },
    {
        "role": "user",
        "content": "Can you make it a couple of degrees warmer in here?",
    },
]

live_data, live_error = run_live_chat(part1_messages, tools=thermostat_tools)

if live_error:
    print("Live tool planning skipped or failed:")
    print(live_error)
else:
    tool_calls = extract_tool_calls(live_data)
    summary = [
        {"name": call["function"]["name"], "arguments": call["function"]["arguments"]}
        for call in tool_calls
    ]

    print("Model:", TARGET_OLLAMA_MODEL)
    print("Assistant text:", extract_message_text(live_data) or "[tool call only]")
    print()
    print("Tool calls returned:")
    show_json(summary or ["[none]"])
    print()
    print("Prompt tokens:", live_data.get("prompt_eval_count"))
    print("Generated tokens:", live_data.get("eval_count"))

Model: minimax-m2.5:cloud
Assistant text: [tool call only]

Tool calls returned:
[
  {
    "name": "get_room_temp",
    "arguments": {}
  }
]

Prompt tokens: 231
Generated tokens: 54


**Live-model interpretation**
- Treat the live response as evidence, not authority.
- If `minimax-m2.5:cloud` skips a check or emits questionable arguments, that is exactly why the application layer must validate tool names, arguments, and approvals.

### 1.4 Errors and dangerous tools must stay visible

Two rules matter here:
- failed tool calls should stay visible in the transcript;
- instructions inside a tool description are not a safety boundary.

The next cell shows both cases explicitly.

In [12]:
bad_argument_call = {
    "function": {"name": "set_room_temp", "arguments": {"temp": "freezing"}}
}
unknown_tool_call = {"function": {"name": "open_window", "arguments": {}}}

print("Bad argument call:")
show_json(execute_tool_call(bad_argument_call, approved=True))
print()
print("Unknown tool call:")
show_json(execute_tool_call(unknown_tool_call))

Bad argument call:
{
  "status": "error",
  "role": "tool",
  "name": "set_room_temp",
  "content": "ERROR: TypeError: `temp` must be an integer in Fahrenheit."
}

Unknown tool call:
{
  "status": "error",
  "role": "tool",
  "name": "open_window",
  "content": "Unknown tool: open_window"
}


### <b style="color:orange">Task 1: Write an approval policy for a small student-services agent</b>

Treat `lookup_fee_balance` and `get_exam_schedule` as read-only tools.
Treat `drop_course` and `issue_refund` as state-changing tools that must wait for human approval.

Write a policy function that returns:
- `ALLOW` for read-only tools;
- `ASK_USER` for dangerous tools;
- `REJECT` for anything unknown.

In [14]:
task1_tool_calls = [
    {"function": {"name": "lookup_fee_balance", "arguments": {"student_id": "S1234567"}}},
    {
        "function": {
            "name": "drop_course",
            "arguments": {"student_id": "S1234567", "course_code": "COMP4146"},
        }
    },
    {"function": {"name": "issue_refund", "arguments": {"student_id": "S1234567", "amount": 500}}},
    {"function": {"name": "delete_student_record", "arguments": {"student_id": "S1234567"}}},
]

READ_ONLY_TOOLS = {"lookup_fee_balance", "get_exam_schedule"}
DANGEROUS_TOOLS = {"drop_course", "issue_refund"}


def approval_policy(tool_call):
    # TODO: Implement a simple approval policy based on the tool name and arguments.
    name=tool_call["function"]["name"]
    if name in READ_ONLY_TOOLS:
        return "approved"
    if name in DANGEROUS_TOOLS:
        return "ask for approval"
    
    return "rejected"


task1_decisions = [
    {"tool": call["function"]["name"], "decision": approval_policy(call)}
    for call in task1_tool_calls
]

show_json(task1_decisions)


[
  {
    "tool": "lookup_fee_balance",
    "decision": "approved"
  },
  {
    "tool": "drop_course",
    "decision": "ask for approval"
  },
  {
    "tool": "issue_refund",
    "decision": "ask for approval"
  },
  {
    "tool": "delete_student_record",
    "decision": "rejected"
  }
]


---

## Part 2. ReAct: Think, Act, Observe

This part focuses on the ReAct loop:
- plan the next step;
- call a tool;
- read the observation;
- stop only when the evidence is enough.

We keep the environment tiny so the trajectory format stays easy to inspect.

### 2.1 A tiny HotpotQA-style environment

The original ReAct examples use three tools:
- `Search[entity]`
- `Lookup[keyword]`
- `Finish[answer]`

We will implement the same interface over a tiny in-memory corpus containing one classic comparison question plus one extra example for practice.

In [15]:
# The environment is deliberately tiny so every observation is easy to inspect.
WIKI_PAGES = {
    "Arthur's Magazine": [
        "Arthur's Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century.",
        "It was edited by T. S. Arthur.",
    ],
    "First for Women": [
        "First for Women is a women's magazine published in the United States.",
        "The magazine was started in 1989.",
    ],
    "Dune": [
        "Dune is a science fiction novel by Frank Herbert.",
        "It was first published in 1965.",
    ],
    "Neuromancer": [
        "Neuromancer is a science fiction novel by William Gibson.",
        "It was first published in 1984.",
    ],
}

magazine_question = "Which magazine was started first Arthur's Magazine or First for Women?"

In [16]:
class MiniWikiEnv:
    # Small local environment with the same three action names as a ReAct prompt.
    def __init__(self, pages):
        self.pages = pages
        self.current_page = None
        self.finished = False
        self.answer = None

    def reset(self):
        self.current_page = None
        self.finished = False
        self.answer = None

    def search(self, entity):
        if entity in self.pages:
            self.current_page = entity
            return " ".join(self.pages[entity][:2])
        matches = [name for name in self.pages if entity.lower() in name.lower()]
        return "No exact match. Similar entities: " + ", ".join(matches[:5])

    def lookup(self, keyword):
        if self.current_page is None:
            return "No current page. Use Search[...] first."
        for sentence in self.pages[self.current_page]:
            if keyword.lower() in sentence.lower():
                return sentence
        return f"No sentence containing '{keyword}' found in {self.current_page}."

    def finish(self, answer):
        self.finished = True
        self.answer = answer
        return f"Finished with answer: {answer}"

    def step(self, action):
        match = re.fullmatch(r"(Search|Lookup|Finish)\[(.*)\]", action.strip())
        if not match:
            return f"Invalid action format: {action}"
        tool, arg = match.groups()
        if tool == "Search":
            return self.search(arg)
        if tool == "Lookup":
            return self.lookup(arg)
        return self.finish(arg)

In [17]:
react_preamble = dedent(
    '''
    Solve a question answering task with interleaving Thought, Action, and Observation steps.
    Thought can reason about the current situation.
    Action can be Search[entity], Lookup[keyword], or Finish[answer].
    '''
).strip()

env = MiniWikiEnv(WIKI_PAGES)

print(react_preamble)
print()
print("Search demo:")
print(env.step("Search[Arthur's Magazine]"))
print("Lookup demo:")
print(env.step("Lookup[1844]"))

Solve a question answering task with interleaving Thought, Action, and Observation steps.
Thought can reason about the current situation.
Action can be Search[entity], Lookup[keyword], or Finish[answer].

Search demo:
Arthur's Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century. It was edited by T. S. Arthur.
Lookup demo:
Arthur's Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century.


### 2.2 A live model can propose the next ReAct step

The next cell asks `minimax-m2.5:cloud` for one short `Thought 1` and `Action 1`. This does not replace the deterministic environment; it simply shows how a live model might start the loop.

In [18]:
part2_messages = [
    {
        "role": "system",
        "content": (
            "You solve short QA tasks with interleaving Thought and Action steps. "
            "Allowed actions: Search[entity], Lookup[keyword], Finish[answer]. "
            "Keep the response brief."
        ),
    },
    {
        "role": "user",
        "content": (
            f"Question: {magazine_question}\n"
            "Return exactly two lines:\n"
            "Thought 1: <brief reasoning>\n"
            "Action 1: <one action>"
        ),
    },
]

live_data, live_error = run_live_chat(part2_messages)

if live_error:
    print("Live ReAct suggestion skipped or failed:")
    print(live_error)
else:
    print("Model:", TARGET_OLLAMA_MODEL)
    print(extract_message_text(live_data))

Model: minimax-m2.5:cloud
Thought 1: I need to find the founding dates of both Arthur's Magazine and First for Women to determine which was started first.

Action 1: Search[Arthur's Magazine founding date]


### 2.3 Worked ReAct trajectory for the magazine question

This trajectory follows the standard pattern:
1. identify the missing information;
2. search the first entity;
3. interpret the observation;
4. search the second entity;
5. compare the evidence and finish.

In [19]:
ROLE_PREFIX = {
    "question": "Question",
    "thought": "Thought",
    "action": "Action",
    "observation": "Observation",
}


def render_trajectory(traj):
    # Render a trajectory in the same top-to-bottom format students usually read.
    counts = {"thought": 0, "action": 0, "observation": 0}
    lines = []
    for step in traj:
        role = step["role"]
        if role == "question":
            lines.append(f"Question: {step['content']}")
            continue
        counts[role] += 1
        lines.append(f"{ROLE_PREFIX[role]} {counts[role]}: {step['content']}")
    return "\n".join(lines)


def build_trajectory(question, thought_action_pairs):
    # Run a hand-written ReAct plan through the local environment.
    local_env = MiniWikiEnv(WIKI_PAGES)
    trajectory = [{"role": "question", "content": question}]
    for thought, action in thought_action_pairs:
        trajectory.append({"role": "thought", "content": thought})
        trajectory.append({"role": "action", "content": action})
        observation = local_env.step(action)
        if action.startswith("Finish["):
            break
        trajectory.append({"role": "observation", "content": observation})
    return trajectory, local_env.answer

In [20]:
magazine_plan = [
    (
        "I should search Arthur's Magazine and First for Women, then compare the years.",
        "Search[Arthur's Magazine]",
    ),
    (
        "Arthur's Magazine started in 1844. Now I need First for Women.",
        "Search[First for Women]",
    ),
    (
        "First for Women was started in 1989. 1844 < 1989, so Arthur's Magazine was started first.",
        "Finish[Arthur's Magazine]",
    ),
]

magazine_trajectory, magazine_answer = build_trajectory(magazine_question, magazine_plan)

print(render_trajectory(magazine_trajectory))
print()
print("Final answer:", magazine_answer)


Question: Which magazine was started first Arthur's Magazine or First for Women?
Thought 1: I should search Arthur's Magazine and First for Women, then compare the years.
Action 1: Search[Arthur's Magazine]
Observation 1: Arthur's Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century. It was edited by T. S. Arthur.
Thought 2: Arthur's Magazine started in 1844. Now I need First for Women.
Action 2: Search[First for Women]
Observation 2: First for Women is a women's magazine published in the United States. The magazine was started in 1989.
Thought 3: First for Women was started in 1989. 1844 < 1989, so Arthur's Magazine was started first.
Action 3: Finish[Arthur's Magazine]

Final answer: Arthur's Magazine


### <b style="color:orange">Task 2: Write a ReAct trajectory for a new comparison question</b>

Use the same local tools to solve a new question:
- search the first entity;
- search the second entity;
- compare the evidence;
- end with `Finish[...]`.

In [22]:
task2_question = "Which novel was published first Dune or Neuromancer?"

# TODO: Write a ReAct plan for this question and run it through the environment with build_trajectory.
task2_plan = [
    ("I should search Dune and Neuromancer, then compare the years.",
     "Search[Dune]"),
    ("Dune was published in 1965. Now I need Neuromancer.",
    "Search[Neuromancer]"),
    ("Neuromancer was published in 1984. 1965 < 1984, so Dune was published first.",
    "Finish[Dune]"),
]

task2_trajectory, task2_answer = build_trajectory(task2_question, task2_plan)

print(render_trajectory(task2_trajectory))
print()
print("Final answer:", task2_answer)


Question: Which novel was published first Dune or Neuromancer?
Thought 1: I should search Dune and Neuromancer, then compare the years.
Action 1: Search[Dune]
Observation 1: Dune is a science fiction novel by Frank Herbert. It was first published in 1965.
Thought 2: Dune was published in 1965. Now I need Neuromancer.
Action 2: Search[Neuromancer]
Observation 2: Neuromancer is a science fiction novel by William Gibson. It was first published in 1984.
Thought 3: Neuromancer was published in 1984. 1965 < 1984, so Dune was published first.
Action 3: Finish[Dune]

Final answer: Dune


---

## Part 3. Context for Task-Based Interactions

Agent context has four moving parts:
- preamble;
- prior conversation;
- current exchange;
- final agent response.

The main engineering question is not "Can I include more?" but "Which context helps enough to justify its cost?"

### 3.1 Anatomy of agent context

We will build a miniature travel-assistant context with:
- a system preamble;
- earlier user and assistant turns;
- attached artifacts;
- tool traces from the current exchange.

In [23]:
preamble = {
    "role": "system",
    "content": (
        "You are a helpful travel assistant. Use the available tools when needed. "
        "Current date: 2026-03-19."
    ),
}

prior_conversation = [
    {"role": "user", "content": "I need a flight from Hong Kong to Tokyo next Friday."},
    {
        "role": "assistant",
        "content": "I found three candidate flights and attached a short summary.",
    },
]

In [24]:
artifacts = [
    {
        "id": "flight_search_summary",
        "topic": "tokyo_trip",
        "summary": "JL5441 costs USD 350 and stops in ORD; JL022 is nonstop but USD 540; CX520 is USD 470.",
        "tokens": 32,
    },
    {
        "id": "passport_note",
        "topic": "profile",
        "summary": "Traveler passport expires in 2028.",
        "tokens": 8,
    },
]

current_exchange = [
    {"role": "user", "content": "Are there any tickets available for the first one?"},
    {
        "role": "assistant",
        "tool_calls": [
            {"function": {"name": "get_ticket_info", "arguments": {"flight_num": "JL5441"}}}
        ],
    },
    {
        "role": "tool",
        "name": "get_ticket_info",
        "content": json.dumps(
            {
                "price": 350,
                "currency": "USD",
                "stops": ["ORD"],
                "duration": "7h40m",
                "seats": "available",
            }
        ),
    },
]

assistant_response = {
    "role": "assistant",
    "content": "Yes. The first option is available for USD 350 with one stop in Chicago.",
}

In [25]:
artifact_candidates = [
    {
        "id": "flight_search_summary",
        "topic": "tokyo_trip",
        "tokens": 32,
        "utility": 0.96,
        "summary": "Three candidate flights, prices, and stop patterns.",
    },
    {
        "id": "fare_rules",
        "topic": "tokyo_trip",
        "tokens": 42,
        "utility": 0.78,
        "summary": "Fare class restrictions and change penalties.",
    },
    {
        "id": "hotel_loyalty_note",
        "topic": "separate_hotel_task",
        "tokens": 28,
        "utility": 0.31,
        "summary": "Hotel loyalty number and room preference.",
    },
    {
        "id": "passport_note",
        "topic": "profile",
        "tokens": 8,
        "utility": 0.55,
        "summary": "Traveler passport expires in 2028.",
    },
    {
        "id": "refund_policy",
        "topic": "tokyo_trip",
        "tokens": 26,
        "utility": 0.64,
        "summary": "Tickets are refundable within 24 hours for a fee.",
    },
]

In [26]:
def render_artifact_block(artifact_list):
    # Render artifacts as a compact prompt section.
    lines = ["## Attached Data"]
    for artifact in artifact_list:
        lines.append(f"- {artifact['id']}: {artifact['summary']}")
    return "\n".join(lines)


def assemble_context(preamble, prior_conversation, artifact_list, current_exchange):
    # Assemble a simple prompt-like context document for one current exchange.
    sections = ["### Preamble\n" + preamble["content"]]

    if prior_conversation:
        prior_text = "\n".join(
            f"- {message['role']}: {message['content']}" for message in prior_conversation
        )
        sections.append("### Prior Conversation\n" + prior_text)

    if artifact_list:
        sections.append(render_artifact_block(artifact_list))

    current_lines = []
    for message in current_exchange:
        if message["role"] == "assistant" and "tool_calls" in message:
            call = message["tool_calls"][0]["function"]
            current_lines.append(f"- assistant tool call: {call['name']}({call['arguments']})")
        elif message["role"] == "tool":
            current_lines.append(f"- tool {message['name']}: {message['content']}")
        else:
            current_lines.append(f"- {message['role']}: {message['content']}")

    sections.append("### Current Exchange\n" + "\n".join(current_lines))
    return "\n\n".join(sections)

In [27]:
assembled_context = assemble_context(
    preamble, prior_conversation, artifacts, current_exchange
)

print(assembled_context)
print()
print("### Agent Response")
print(assistant_response["content"])

### Preamble
You are a helpful travel assistant. Use the available tools when needed. Current date: 2026-03-19.

### Prior Conversation
- user: I need a flight from Hong Kong to Tokyo next Friday.
- assistant: I found three candidate flights and attached a short summary.

## Attached Data
- flight_search_summary: JL5441 costs USD 350 and stops in ORD; JL022 is nonstop but USD 540; CX520 is USD 470.
- passport_note: Traveler passport expires in 2028.

### Current Exchange
- user: Are there any tickets available for the first one?
- assistant tool call: get_ticket_info({'flight_num': 'JL5441'})
- tool get_ticket_info: {"price": 350, "currency": "USD", "stops": ["ORD"], "duration": "7h40m", "seats": "available"}

### Agent Response
Yes. The first option is available for USD 350 with one stop in Chicago.


### 3.2 A live model can help judge artifact relevance

The next cell asks `minimax-m2.5:cloud` to identify which artifacts look most relevant for the current travel question. This is only a helper signal, not the final selection policy.

In [28]:
artifact_text = "\n".join(
    f"- {artifact['id']}: {artifact['summary']}" for artifact in artifact_candidates
)

part3_messages = [
    {
        "role": "system",
        "content": (
            "You help an agent choose context. "
            "Return exactly three short lines. "
            "Each line must begin with KEEP: and use one provided artifact id."
        ),
    },
    {
        "role": "user",
        "content": (
            "Active task: answer whether tickets are available for the first Tokyo flight.\n"
            f"Candidate artifacts:\n{artifact_text}\n\n"
            "Choose the three most useful artifacts.\n"
            "Return exactly three lines in this format:\n"
            "KEEP: <id> - <short reason>"
        ),
    },
]

live_data, live_error = run_live_chat(part3_messages)

if live_error:
    print("Live context selection skipped or failed:")
    print(live_error)
else:
    print("Model:", TARGET_OLLAMA_MODEL)
    print(extract_message_text(live_data))

Model: minimax-m2.5:cloud
KEEP: flight_search_summary - Contains flight options to Tokyo with prices and availability
KEEP: fare_rules - Shows fare class restrictions that affect ticket availability
KEEP: refund_policy - Provides ticket purchase and refund terms relevant to buying


### 3.3 Context assembly is a trade-off

Ask four questions:
- which tools are needed right now?
- which artifacts matter for the current task?
- how should artifacts be represented?
- what can be compressed, summarized, or dropped?

The next cell uses a simple scoring rule to keep only topical artifacts under a budget.

In [29]:
def score_artifact(artifact, active_topic, keep_topics=frozenset({"profile"})):
    # Simple hand-written relevance score used as a transparent baseline.
    if artifact["topic"] == active_topic:
        topical_bonus = 1.0
    elif artifact["topic"] in keep_topics:
        topical_bonus = 0.35
    else:
        topical_bonus = -1.0
    return topical_bonus + artifact["utility"] - 0.003 * artifact["tokens"]


def select_artifacts(artifact_list, active_topic, budget):
    # Greedily keep only useful artifacts that still fit the token budget.
    ranked = sorted(
        artifact_list,
        key=lambda artifact: score_artifact(artifact, active_topic),
        reverse=True,
    )
    selected = []
    used = 0
    for artifact in ranked:
        if score_artifact(artifact, active_topic) < 0:
            continue
        if used + artifact["tokens"] > budget:
            continue
        selected.append(artifact)
        used += artifact["tokens"]
    return selected, used


selected_artifacts, used_tokens = select_artifacts(
    artifact_candidates, active_topic="tokyo_trip", budget=70
)

print("Selected artifact IDs:", [artifact["id"] for artifact in selected_artifacts])
print("Approx token budget used:", used_tokens)
print()
print(render_artifact_block(selected_artifacts))

Selected artifact IDs: ['flight_search_summary', 'refund_policy', 'passport_note']
Approx token budget used: 66

## Attached Data
- flight_search_summary: Three candidate flights, prices, and stop patterns.
- refund_policy: Tickets are refundable within 24 hours for a fee.
- passport_note: Traveler passport expires in 2028.


### <b style="color:orange">Task 3: Select the right artifacts for a refund request</b>

A travel agent is now handling a **refund** question rather than a search question.

Build the current artifact set so that it:
- prioritizes artifacts about the active topic `refund`;
- keeps lightweight profile data if it is still helpful;
- drops unrelated artifacts even if they are short.

In [31]:
task3_artifacts = [
    {
        "id": "refund_policy",
        "topic": "refund",
        "tokens": 26,
        "utility": 0.92,
        "summary": "Tickets are refundable within 24 hours minus a USD 30 fee.",
    },
    {
        "id": "original_booking",
        "topic": "refund",
        "tokens": 18,
        "utility": 0.88,
        "summary": "Booking JL5441 was purchased 6 hours ago for USD 350.",
    },
    {
        "id": "seat_map",
        "topic": "seat_selection",
        "tokens": 22,
        "utility": 0.15,
        "summary": "12A is an aisle seat.",
    },
    {
        "id": "passport_note",
        "topic": "profile",
        "tokens": 8,
        "utility": 0.40,
        "summary": "Traveler passport expires in 2028.",
    },
    {
        "id": "weather_alert",
        "topic": "weather",
        "tokens": 14,
        "utility": 0.20,
        "summary": "Light rain expected in Tokyo next Friday.",
    },
]

# TODO: Run your artifact selection function on this new list and see how the output changes. Hint: you can reuse the same function since it's designed to be general.
selected_artifacts, used_tokens = select_artifacts(
    task3_artifacts, active_topic="refund", budget=70
)
print("Selected artifact IDs:", [artifact["id"] for artifact in selected_artifacts])
print("Approx token budget used:", used_tokens)
print()
print(render_artifact_block(selected_artifacts))

Selected artifact IDs: ['refund_policy', 'original_booking', 'passport_note']
Approx token budget used: 52

## Attached Data
- refund_policy: Tickets are refundable within 24 hours minus a USD 30 fee.
- original_booking: Booking JL5441 was purchased 6 hours ago for USD 350.
- passport_note: Traveler passport expires in 2028.


---

## Part 4. From `process_messages()` to `run_conversation()`

Once a single message can request tools and consume results, you are one loop away from a conversational agent.

We will build a tiny scripted assistant to expose the control flow:
1. append the user turn;
2. let the assistant request tools;
3. execute tools through the application;
4. append the results;
5. surface the final assistant reply;
6. persist the transcript for the next turn.

### 4.1 A deterministic conversation loop

This is a mock planner, not a live LLM. The goal is to make the control flow visible:
- the assistant can either speak or request a tool;
- the application executes the tool and appends the result;
- dangerous tools wait for approval before execution.

In [26]:
def execute_conversation_tool(tool_call, state, approved=False):
    # Execute one tool request inside the scripted conversation loop.
    name = tool_call["function"]["name"]
    arguments = tool_call["function"].get("arguments", {})

    if name == "get_room_temp":
        return {"status": "ok", "role": "tool", "name": name, "content": str(state["temp"])}

    if name == "set_room_temp":
        if not approved:
            return {
                "status": "awaiting_approval",
                "role": "tool",
                "name": name,
                "content": f"Approval required for {name}({arguments})",
            }
        state["temp"] = arguments["temp"]
        return {"status": "ok", "role": "tool", "name": name, "content": "DONE"}

    return {"status": "error", "role": "tool", "name": name, "content": f"Unknown tool: {name}"}

In [27]:
def mock_home_assistant(messages, state):
    # Small scripted planner used only to make the loop easy to inspect.
    last = messages[-1]

    if last["role"] == "user":
        text = last["content"].lower()
        if "cool this place down" in text or "hot in here" in text:
            return {
                "role": "assistant",
                "content": None,
                "tool_calls": [{"function": {"name": "get_room_temp", "arguments": {}}}],
            }
        if "lots cooler" in text:
            return {
                "role": "assistant",
                "content": None,
                "tool_calls": [{"function": {"name": "set_room_temp", "arguments": {"temp": 50}}}],
            }
        if "put it back" in text:
            return {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "function": {
                            "name": "set_room_temp",
                            "arguments": {"temp": state["original_temp"]},
                        }
                    }
                ],
            }
        return {"role": "assistant", "content": "I can help with temperature changes in this demo."}

    if last["role"] == "tool" and last["name"] == "get_room_temp":
        temp = int(last["content"])
        return {
            "role": "assistant",
            "content": (
                f"The current room temperature is {temp}F, which is actually quite cool. "
                "If you still want it cooler, tell me how much."
            ),
        }

    if last["role"] == "tool" and last["name"] == "set_room_temp":
        return {"role": "assistant", "content": f"I've set the room temperature to {state['temp']}F."}

    return {"role": "assistant", "content": "Waiting for the next user turn."}

In [28]:
def pretty_print_transcript(messages):
    # Print the conversation in a tool-aware transcript format.
    for message in messages:
        if message["role"] == "assistant" and message.get("tool_calls"):
            call = message["tool_calls"][0]["function"]
            print(f"assistant -> tool: {call['name']}({call['arguments']})")
        elif message["role"] == "tool":
            print(f"tool {message['name']} -> {message['content']}")
        else:
            print(f"{message['role']}: {message['content']}")


def run_conversation(user_turns, approvals):
    # Drive the whole deterministic interaction turn by turn.
    state = {"temp": 64, "original_temp": 64}
    messages = [{"role": "system", "content": "You are HomeBoy, a helpful home assistant."}]

    for turn_index, user_text in enumerate(user_turns, start=1):
        messages.append({"role": "user", "content": user_text})

        while True:
            assistant_message = mock_home_assistant(messages, state)
            messages.append(assistant_message)

            if not assistant_message.get("tool_calls"):
                break

            call = assistant_message["tool_calls"][0]
            key = (turn_index, call["function"]["name"])
            tool_result = execute_conversation_tool(
                call, state, approved=approvals.get(key, False)
            )
            messages.append(tool_result)

            if tool_result["status"] != "ok":
                break

    return messages

In [29]:
user_turns = [
    "Golly gee, it's hot in here. Let's cool this place down.",
    "Make it LOTS cooler.",
    "Whoa... that's a bit extreme. Put it back to where it started.",
]
approvals = {
    (2, "set_room_temp"): True,
    (3, "set_room_temp"): True,
}

transcript = run_conversation(user_turns, approvals)
pretty_print_transcript(transcript)

system: You are HomeBoy, a helpful home assistant.
user: Golly gee, it's hot in here. Let's cool this place down.
assistant -> tool: get_room_temp({})
tool get_room_temp -> 64
assistant: The current room temperature is 64F, which is actually quite cool. If you still want it cooler, tell me how much.
user: Make it LOTS cooler.
assistant -> tool: set_room_temp({'temp': 50})
tool set_room_temp -> DONE
assistant: I've set the room temperature to 50F.
user: Whoa... that's a bit extreme. Put it back to where it started.
assistant -> tool: set_room_temp({'temp': 64})
tool set_room_temp -> DONE
assistant: I've set the room temperature to 64F.


### 4.2 A minimal live `process_messages()` example

The next two functions use the same compact pattern as the earlier thermostat example:
- `process_messages(client, messages)` sends the transcript to the model, appends the assistant message, executes any requested tools, and appends tool outputs;
- `run_conversation(client)` initializes the system message, collects user input, and loops until the assistant is ready for the next user turn.

The model call goes through Ollama with **`minimax-m2.5:cloud`**. In interactive use, enter a blank line to stop. During batch execution, the notebook falls back to one short sample turn.


In [30]:
demo_user_inputs = [
    "Can you make it a couple of degrees warmer in here?",
    "Please put it back to 74F.",
    "",
]


def simple_input(prompt):
    # Use real notebook input when available; otherwise fall back to a short demo.
    try:
        return input(prompt)
    except Exception as exc:
        value = demo_user_inputs.pop(0) if demo_user_inputs else ""
        shown = value if value else "[blank]"
        print(f"{prompt}{shown} (sample used because {type(exc).__name__} blocked input)")
        return value


In [31]:
def process_messages(client, messages):
    # Send the transcript to the model and append any resulting tool calls and tool outputs.
    if client is None or not OLLAMA_READY:
        messages.append(
            {"role": "assistant", "content": f"[live backend unavailable] {OLLAMA_STATUS}"}
        )
        return messages

    response = client.chat(
        model=TARGET_OLLAMA_MODEL,
        messages=messages,
        tools=thermostat_tools,
        options={"temperature": 0},
    )
    response_message = to_builtin(response)["message"]
    messages.append(response_message)

    if response_message.get("tool_calls"):
        for tool_call in response_message["tool_calls"]:
            function_name = tool_call["function"]["name"]
            function_to_call = available_functions[function_name]
            function_args = tool_call["function"].get("arguments", {})
            if isinstance(function_args, str):
                function_args = json.loads(function_args)
            function_response = function_to_call(**function_args)
            messages.append(
                {
                    "tool_call_id": tool_call.get("id", function_name),
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )
    return messages


In [32]:
def run_conversation(client):
    # Manage the full conversation state one user turn at a time.
    messages = [
        {
            "role": "system",
            "content": "You are a helpful thermostat assistant.",
        }
    ]

    while True:
        user_input = simple_input(">> ")
        if user_input == "":
            break
        messages.append({"role": "user", "content": user_input})

        while True:
            start_index = len(messages)
            new_messages = process_messages(client, messages)

            for message in new_messages[start_index:]:
                if message["role"] == "assistant" and message.get("tool_calls"):
                    for tool_call in message["tool_calls"]:
                        function = tool_call["function"]
                        print(
                            f"assistant -> tool: {function['name']}({function.get('arguments', {})})"
                        )
                elif message["role"] == "tool":
                    print(f"tool {message['name']} -> {message['content']}")

            last_message = new_messages[-1]
            if last_message["role"] == "tool":
                continue

            if last_message.get("content") is not None:
                print(last_message["content"])
                if not last_message.get("tool_calls"):
                    break
    return messages


**Suggested Conversation Script**

Try this short script if you want a predictable tool-using exchange:
1. `Can you make it a couple of degrees warmer in here?`
2. `Please put it back to 74F.`
3. Enter a blank line to stop.


In [33]:
thermostat_state["temp"] = 74
thermostat_state["original_temp"] = 74
demo_user_inputs[:] = [
    "Can you make it a couple of degrees warmer in here?",
    "Please put it back to 74F.",
    "",
]

conversation_messages = run_conversation(OLLAMA_CLIENT)
print("Final thermostat state:", thermostat_state)


assistant -> tool: get_room_temp({})
tool get_room_temp -> 74
assistant -> tool: set_room_temp({'temp': 76})
tool set_room_temp -> DONE
Done! I've turned up the temperature by 2 degrees to 76°F. It should feel warmer shortly.
assistant -> tool: set_room_temp({'temp': 74})
tool set_room_temp -> DONE
Done! I've set the temperature back to 74°F.
Final thermostat state: {'temp': 74, 'original_temp': 74}


In [34]:
print("Transcript snapshot:")
pretty_print_transcript(conversation_messages)


Transcript snapshot:
system: You are a helpful thermostat assistant.
user: Can you make it a couple of degrees warmer in here?
assistant -> tool: get_room_temp({})
tool get_room_temp -> 74
assistant -> tool: set_room_temp({'temp': 76})
tool set_room_temp -> DONE
assistant: Done! I've turned up the temperature by 2 degrees to 76°F. It should feel warmer shortly.
user: Please put it back to 74F.
assistant -> tool: set_room_temp({'temp': 74})
tool set_room_temp -> DONE
assistant: Done! I've set the temperature back to 74°F.


## Pitfalls and Extension

**Common pitfall:** treating tool descriptions as if they enforce policy.

**Why it hurts**
- the model may still call the wrong tool or call a valid tool at the wrong time;
- invisible tool errors remove evidence that the model needs for recovery;
- oversized tool sets and artifact dumps confuse routing and waste tokens.

**Better habit**
- enforce validation and approvals in deterministic application code;
- keep tools narrow and arguments simple;
- assemble context intentionally and evaluate the result on fixed tasks.

**Optional extension**
- add another tool, such as `open_window()` or `turn_on_fan()`, then compare the new live behavior against the thermostat-only loop.


## Key Takeaways

1. Tools extend chat into hidden knowledge, live state, and external actions.
2. `minimax-m2.5:cloud` can plan tool calls through Ollama, but the application still owns execution and validation.
3. ReAct combines reasoning with evidence gathering through `Thought`, `Action`, and `Observation`.
4. Successful trajectories can be serialized and reused as supervised finetuning data.
5. Context assembly is a budget, relevance, and safety trade-off.
6. A conversation loop works by repeatedly appending assistant and tool messages to the same transcript.
7. Validation, approvals, transcript management, and UI transparency belong to the application layer.


---

### <b style="color:orange">Submission</b>

Make sure you:
1. execute the notebook from top to bottom;
2. complete each task;
3. submit your finished notebook as `YourName_YourStudentID.ipynb` on Moodle.